# 03 — Clean-Room SPAN (1.7 MB) & Real-PLKSR (15 MB) Super-Res / DeJPEG

Retrains Apache-2.0 `hongyuanyu/SPAN` and MIT `dslisleedh/PLKSR` on **PD12M (CC0) + MatSynth (CC0) + Infinite renders** using Charbonnier L1 + **LPIPS (BSD-2)** (zero DIV2K taint).


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
# Clone clean Apache-2.0 / MIT model definitions
!git clone -q https://github.com/hongyuanyu/SPAN.git /content/SPAN
!git clone -q https://github.com/dslisleedh/PLKSR.git /content/PLKSR
import lpips
loss_lpips = lpips.LPIPS(net='alex').cuda()  # BSD-2-Clause perceptual loss
print("Ready: SPAN + PLKSR + LPIPS (BSD-2). Stream PD12M CC0 crops -> random JPEG Q35-95 + bicubic 2x/4x degradation.")


In [ ]:
def export_infinite_onnx(model, dummy, out_path, input_names, output_names, fp16=True):
    model.eval().cpu()
    tmp_fp32 = out_path.replace(".onnx", "_fp32.onnx")
    torch.onnx.export(
        model, dummy, tmp_fp32, opset_version=17, do_constant_folding=True,
        input_names=input_names, output_names=output_names,
    )
    m = onnxslim.slim(tmp_fp32)
    onnx.checker.check_model(m)
    if fp16:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        m = convert_float_to_float16(m, keep_io_types=True)
        onnx.checker.check_model(m)
    onnx.save(m, out_path)
    data = open(out_path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    print(f"Exported {out_path}: {len(data)/1e6:.2f} MB  sha256={sha}")
    return sha

# After training: call model.switch_to_deploy() on SPAN before export:
# export_infinite_onnx(span_model, torch.randn(1,3,512,512), '/content/out/span_2x_clean.onnx', ['input'], ['output'], fp16=True)
# export_infinite_onnx(plksr_model, torch.randn(1,3,512,512), '/content/out/real_plksr_clean.onnx', ['input'], ['output'], fp16=True)
